In [6]:
!uv add google-adk google-generativeai -q
!uv add plotly pandas python-dotenv -q

In [8]:
# Default libs
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid5
from typing import Any, List

# Installed libs
import pandas as pd
import plotly.graph_objects as go
import vertexai
from IPython.display import HTML, Markdown, display

# ADK libs
import google.adk as adk
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService, Session
from google.adk.tools import google_search
from google.genai import types
from google.genai.types import Content, Part

# Load envs
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
print(os.getenv("GOOGLE_CLOUD_PROJECT"))

project-de34b621-684f-4b33-b42


In [10]:
GOOGLE_CLOUD_PROJECT = os.getenv("GOOGLE_CLOUD_PROJECT")

In [15]:
# Add gcloud to PATH for Jupyter
import os
import shutil

# Find gcloud executable
gcloud_path = shutil.which("gcloud")
if gcloud_path:
    gcloud_dir = os.path.dirname(gcloud_path)
    os.environ["PATH"] = gcloud_dir + os.pathsep + os.environ.get("PATH", "")
    print(f"Found gcloud at: {gcloud_path}")
else:
    print("gcloud not found in PATH, trying default locations...")
    # Try common locations
    default_paths = [
        os.path.expanduser("~/.local/bin/gcloud"),
        os.path.expanduser("~/google-cloud-sdk/bin/gcloud"),
        "/usr/local/bin/gcloud",
    ]
    for path in default_paths:
        if os.path.exists(path):
            os.environ["PATH"] = os.path.dirname(path) + os.pathsep + os.environ.get("PATH", "")
            print(f"Found gcloud at: {path}")
            break

gcloud not found in PATH, trying default locations...
Found gcloud at: /home/vasim/google-cloud-sdk/bin/gcloud


In [30]:
# !gcloud auth login
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=PLVML6PDRYs1Y0bJZPpJe6LqSJWUAx&access_type=offline&code_challenge=2gwTf3Uc9XpPxFJHrZG5N1aEFzvhBVatkn-cvpM_Ir0&code_challenge_method=S256

Gtk-Message: 20:28:32.499: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.

Credentials saved to file: [/home/vasim/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "project-de34b621-684f-4b33-b42" was added to ADC which can be used by Google client librarie

In [19]:
!gcloud config set project $GOOGLE_CLOUD_PROJECT

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


In [18]:
!gcloud services enable aiplatform.googleapis.com --project=$GOOGLE_CLOUD_PROJECT

Install gcloud cli if it is not available:
```shell
# Download and install gcloud SDK
curl https://sdk.cloud.google.com | bash
source ~/.bashrc
```

---
## Part 1: Your First Agent - The Day Trip Genie 🧞

Meet your first creation! The `day_trip_agent` is a simple but powerful assistant. We're making it a little smarter by teaching it to understand **budget constraints**.

* **Agent**: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
* **Session**: The conversation history. For this simple agent, it's just a container for a single request-response.
* **Runner**: The engine that connects the `Agent` and the `Session` to process your request and get a response.

#### Flow

```text
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```

In [20]:
def create_day_trip_agent():
    """Creating a day trip agent using ADK"""
    try:
        return Agent(
            name="day_trip_agent", # agent name
            model="gemini-2.5-flash", # choose model here
            description="Adent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.", # description shows agent portfolio
            instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
            tools=[google_search]
        )
    
    except Exception as e:
        print(e)

In [21]:
day_trip_agent = create_day_trip_agent()
print(f"Agent '{day_trip_agent.name}' is created and ready for adventure!")

Agent 'day_trip_agent' is created and ready for adventure!


In [28]:
# Define a helper function to call the agent


async def run_agent_query(
    agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False
):
    """
    Initializes a runner and executes a query for a given agent and session.
    """
    print(f"\n Running query for agent: '{agent.name}' in session: '{session.id}'...")

    # Define a runner
    runner = Runner(agent=agent, session_service=session_service, app_name=agent.name)

    # Call
    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user"),
        ):
            if not is_router:
                # Let's see what agent is  thinking...
                print(f"EVENT: {event}")

            if event.is_final_response():
                final_response = event.content.parts[0].text

    except Exception as e:
        raise e

    if not is_router:
        print("\n" + "-" * 50)
        print("Final response:\n")
        display(Markdown(final_response))
        print("-" * 50 + "\n")

    return final_response


# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [31]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

 Running query for agent: 'day_trip_agent' in session: '3b4c0628-3c21-46d1-89ba-85027f5c3b41'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Get ready for a day of artistic exploration and tranquility near Sunnyvale, California, all while keeping your budget in mind! This itinerary focuses on free and affordable cultural experiences and natural beauty.

Here's your relaxing and artsy day trip:

## Relaxing & Artsy Day Trip near Sunnyvale, CA

**Budget:** Affordable (primarily free activities, with options for affordable meals and paid parking if applicable).

### Morning: Immerse in Art at Stanford (10:30 AM - 1:30 PM)

*   **10:30 AM - Arrive at Cantor Arts Center, Stanford University:** Start your day with a visit to the **Cantor Arts Center** at Stanford University. Admission to this renowned art museum is always free. Explore its diverse collec

Get ready for a day of artistic exploration and tranquility near Sunnyvale, California, all while keeping your budget in mind! This itinerary focuses on free and affordable cultural experiences and natural beauty.

Here's your relaxing and artsy day trip:

## Relaxing & Artsy Day Trip near Sunnyvale, CA

**Budget:** Affordable (primarily free activities, with options for affordable meals and paid parking if applicable).

### Morning: Immerse in Art at Stanford (10:30 AM - 1:30 PM)

*   **10:30 AM - Arrive at Cantor Arts Center, Stanford University:** Start your day with a visit to the **Cantor Arts Center** at Stanford University. Admission to this renowned art museum is always free. Explore its diverse collection, which spans 5,000 years of art history and includes an impressive array of Auguste Rodin sculptures, many displayed in the outdoor **Rodin Sculpture Garden**. The museum is open on Thursdays (April 3, 2026) from 11:00 a.m. to 8:00 p.m. Take your time wandering through the galleries and enjoying the serene outdoor spaces on the beautiful Stanford campus.
    *   *Cost: Free admission to the museum. Parking on weekdays is paid and managed via ParkMobile.*

### Lunch: Affordable Bites & Park Relaxation (1:30 PM - 2:30 PM)

*   **1:30 PM - Grab Affordable Lunch:** Head back towards Sunnyvale or find a quick, affordable bite near Stanford. Consider popular and inexpensive options like **Cam Hung** for banh mi or a local taco truck such as **El Califas Taco Truck** or **Taqueria Tarasco** in Sunnyvale for delicious and budget-friendly Mexican food.
*   **2:00 PM - Picnic in the Park:** Take your lunch to a nearby park for a relaxing picnic. **Washington Park** in Sunnyvale is a great choice, and you can even explore some of the city's public art installations there.

### Afternoon: Self-Guided Public Art & Nature Stroll (2:30 PM - 5:30 PM)

*   **2:30 PM - Sunnyvale Public Art Walking Tour:** Discover Sunnyvale's vibrant public art scene with a self-guided walking tour. The city boasts over 200 public art pieces, with several themed routes available. Focus on a specific area like the **Civic Center and Washington Park** tour or the **Downtown and Murphy Park** tour to enjoy art integrated into the urban landscape. Look out for the "Sun Flair" sculptures, a new public art program featuring uniquely transformed sun sculptures in various parks, which began installation in early 2025. This activity combines relaxation with appreciating local creativity.
    *   *Cost: Free.*

### Evening: Artsy Evening in San Jose (6:00 PM - 8:00 PM)

*   **6:00 PM - San Jose Museum of Art "First Friday" Event:** Conclude your artsy day at the **San Jose Museum of Art (SJMA)**. Conveniently, tomorrow, April 3, 2026, is a "First Friday," meaning you can enjoy free admission after 6:00 p.m. SJMA focuses on modern and contemporary art and often hosts special programs. On April 3rd, they are featuring a special screening of "With Drawn Arms" as part of their First Friday event. Explore the exhibitions and take advantage of this free cultural offering.
    *   *Cost: Free admission after 6:00 p.m. on First Fridays. Standard adult admission is usually $20, but currently reduced through April 9.*
    *   *Note: Check for parking options in Downtown San Jose; garages are typically available for a fee.*

This itinerary offers a blend of free museum entry, public art appreciation, and relaxed park time, perfectly aligning with your request for an affordable, relaxing, and artsy day trip.

--------------------------------------------------

